In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from torch.cuda.amp import GradScaler, autocast  # Mixed Precision for speed
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, accuracy_score
import numpy as np
import copy
import os

# ---------------- CONFIGURATION ----------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32         # ResNet-18 is light; 32 or 64 works well
EPOCHS = 10
LR = 1e-3               # Standard starting LR for ResNet
CHECKPOINT_PATH = "best_resnet18.pth"

print(f"🚀 Training Baseline: ResNet-18 on {DEVICE}")

# ---------------- 1. DATA LOADER ----------------
# Reuse your existing MemoryCachedDataset class
class MemoryCachedDataset(Dataset):
    def __init__(self, cache_path):
        data = torch.load(cache_path)
        self.samples = data["samples"]
        self.classes = data["classes"]
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

# Load Data
try:
    train_ds = MemoryCachedDataset("train_cache.pt")
    val_ds = MemoryCachedDataset("val_cache.pt")
    test_ds = MemoryCachedDataset("test_cache.pt")
    NUM_CLASSES = len(train_ds.classes)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
    print(f"Classes: {train_ds.classes}")
except Exception as e:
    print(f"❌ Error loading data: {e}")

# ---------------- 2. MODEL SETUP ----------------
def initialize_resnet18(num_classes):
    print("Initializing ResNet-18...")
    # Load pre-trained ResNet-18
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    
    # Modify the final layer for your specific number of classes
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    
    return model.to(DEVICE)

model = initialize_resnet18(NUM_CLASSES)

# ---------------- 3. OPTIMIZER & LOSS ----------------
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()
scaler = GradScaler()  # For Mixed Precision (FP16)

# ---------------- 4. TRAINING LOOP ----------------
best_val_acc = 0.0

print("Starting Training...")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    # Progress bar
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for imgs, labels in loop:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        # Mixed Precision Forward Pass
        with autocast():
            logits = model(imgs)
            loss = criterion(logits, labels)
        
        # Scaled Backward Pass
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    
    scheduler.step()

    # Validation
    model.eval()
    val_correct = 0; val_total = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            # No autocast needed for validation usually, but keeps it consistent
            logits = model(imgs)
            preds = logits.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
            
    val_acc = val_correct / val_total
    avg_loss = total_loss / len(train_loader)
    
    print(f"Epoch {epoch+1}: Loss={avg_loss:.4f} | Val Acc={val_acc*100:.2f}%")
    
    # Save Best Model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"--> 💾 Saved Best Model ({val_acc*100:.2f}%)")



🚀 Training Baseline: ResNet-18 on cuda
Classes: ['Cysts_Structural', 'Dysarthia', 'Laryngitis', 'Vox senilis', 'parkinson', 'spasmodische_dysphonie']
Initializing ResNet-18...
Starting Training...


Epoch 1/10: 100%|██████████| 216/216 [00:12<00:00, 17.10it/s, loss=0.471]


Epoch 1: Loss=0.6546 | Val Acc=75.78%
--> 💾 Saved Best Model (75.78%)


Epoch 2/10: 100%|██████████| 216/216 [00:11<00:00, 18.48it/s, loss=0.251]


Epoch 2: Loss=0.4175 | Val Acc=75.78%


...

Epoch 8: Loss=0.0072 | Val Acc=90.13%


Epoch 9/10: 100%|██████████| 216/216 [00:11<00:00, 18.60it/s, loss=0.00111] 


Epoch 9: Loss=0.0036 | Val Acc=91.03%
--> 💾 Saved Best Model (91.03%)


Epoch 10/10: 100%|██████████| 216/216 [00:11<00:00, 18.61it/s, loss=0.000981]


Epoch 10: Loss=0.0030 | Val Acc=91.48%
--> 💾 Saved Best Model (91.48%)


In [4]:
# ---------------- 5. FINAL EVALUATION ----------------
print("\n--- EVALUATING ON TEST SET ---")
model.load_state_dict(torch.load(CHECKPOINT_PATH))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

# Calculate Metrics
print(f"Final Test Accuracy: {accuracy_score(all_labels, all_preds)*100:.2f}%")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=train_ds.classes))


--- EVALUATING ON TEST SET ---
Final Test Accuracy: 91.15%

Classification Report:
                        precision    recall  f1-score   support

      Cysts_Structural       0.95      0.86      0.90        22
             Dysarthia       1.00      1.00      1.00        42
            Laryngitis       0.76      0.83      0.80        42
           Vox senilis       0.92      0.94      0.93        50
             parkinson       1.00      1.00      1.00        50
spasmodische_dysphonie       0.76      0.65      0.70        20

              accuracy                           0.91       226
             macro avg       0.90      0.88      0.89       226
          weighted avg       0.91      0.91      0.91       226

